# Chapter 7 &mdash; $Eclosure$: What $\varepsilon$ Edges Do to Simulation

**Concept 6 of the Chapter 7 decomposition:** *$Eclosure$: What $\varepsilon$ Edges Do to Simulation*

Tokens "ooze" along $\varepsilon$ edges one way; $Eclosure(q)$ is everything reachable from $q$ for free.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7-NFA/Concept-Eclosure/Concept-Eclosure.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateNFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateNFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateNFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


With $\varepsilon$ edges present, a token does not sit still: it **oozes** along every
$\varepsilon$ edge it can, *in the direction of the arrow only*.

$$Eclosure(S) = \{q : q \text{ reachable from some } s\in S \text{ by } \varepsilon\text{ edges alone}\}$$

Two properties matter:

* $S \subseteq Eclosure(S)$ &mdash; zero $\varepsilon$ steps is allowed;
* $Eclosure$ is **idempotent**: $Eclosure(Eclosure(S)) = Eclosure(S)$ &mdash; it is a
  least fixed point, computed by iterating until nothing is added.

It is also why $\varepsilon$ **cycles** are harmless.

## 2. Definitions

### A machine with an $\varepsilon$ chain and an $\varepsilon$ cycle

In [ ]:
N = md2mc('''NFA
I : '' -> A
A : '' -> B
B : '' -> A          !! an epsilon CYCLE
B : 0 -> F
A : 1 -> F
''')

### $Eclosure$, computed as a least fixed point

In [ ]:
def eclose(N, S):
    cur = set(S)
    while True:
        nxt = cur | {t for q in cur for t in step_nfa(N, q, '')}
        if nxt == cur: return cur
        cur = nxt

<!-- nav-strip -->

---

&larr;&nbsp;[Ch7&nbsp;5.&nbsp;Simulating an NFA Without $\varepsilon$: Tracking the Set of Token Positions](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7-NFA/Concept-Simulating-Without-Epsilon/Concept-Simulating-Without-Epsilon.ipynb) &nbsp;&middot;&nbsp; [**Chapter 7** index](https://github.com/ganeshutah/Jove/blob/master/Chapter7-NFA/README.md) &nbsp;&middot;&nbsp; [Ch7&nbsp;7.&nbsp;The Language of an NFA: $\hat{\delta}$ via $Eclosure$–$\delta$–$Eclosure$](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7-NFA/Concept-Delta-Hat-Via-Eclosure/Concept-Delta-Hat-Via-Eclosure.ipynb)&nbsp;&rarr;

---

## 3. Tests

Our fixed point agrees with Jove's `Eclosure`.

In [ ]:
for S in [{'I'}, {'A'}, {'B'}, {'F'}, {'I','F'}]:
    mine, jove = eclose(N, S), Eclosure(N, S)
    print("Eclosure(%-9s) = %-18s  agrees: %s"
          % (sorted(S), sorted(jove), mine == jove))
    assert mine == jove

**Reflexive:** $S \subseteq Eclosure(S)$, always.

In [ ]:
for S in [{'I'}, {'B'}, {'F'}]:
    assert S <= Eclosure(N, S)
print("every state is in its own Eclosure -- zero epsilon steps counts")

**Idempotent:** closing twice adds nothing. That is what 'closure' means.

In [ ]:
for S in [{'I'}, {'A'}, {'B'}, {'I','F'}]:
    assert Eclosure(N, Eclosure(N, S)) == Eclosure(N, S)
print("Eclosure(Eclosure(S)) == Eclosure(S) for every set tried")

**Directional:** the arrow matters &mdash; `A` reaches `B`, but the reverse needs its own edge.

In [ ]:
one_way = md2mc('''NFA
I : '' -> A
A : 0 -> F
''')
print("Eclosure({I}) =", sorted(Eclosure(one_way, {'I'})))
print("Eclosure({A}) =", sorted(Eclosure(one_way, {'A'})), " <- does NOT contain I")
assert 'I' not in Eclosure(one_way, {'A'})

The $\varepsilon$ **cycle** terminates because the closure is a least fixed point.

In [ ]:
print("Eclosure({A}) in the cyclic machine :", sorted(Eclosure(N, {'A'})))
print("the loop A -> B -> A adds nothing new on the second pass, so it stops.")
assert Eclosure(N, {'A'}) == Eclosure(N, {'B'})

## 4. Animation

The $\varepsilon$ edges, drawn: tokens spread along them before any symbol is read.

In [ ]:
from jove.AnimateNFA import *
AnimateNFA(N, FuseEdges=True)

## 5. Exercises


1. Compute $Eclosure$ by hand for a chain of five $\varepsilon$ edges.
2. Why must $Eclosure$ be applied *before* the first symbol as well as after each one?
3. What is $Eclosure(\emptyset)$?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter7-NFA/Concept-Eclosure')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')